1. Cài đặt Spark

In [2]:
!java -version
!pip install -q pyspark

[0.018s][warning][os,container] Cgroup memory controller path at '/sys/fs/cgroup' seems to have moved to '/../../jupyter-children', detected limits won't be accurate
[0.018s][warning][os,container] Cgroup cpu controller path at '/sys/fs/cgroup' seems to have moved to '/../../jupyter-children', detected limits won't be accurate
openjdk version "21.0.12.1" 2026-08-18
OpenJDK Runtime Environment (build 21.0.12.1+1-1-24.04.4-Ubuntu)
OpenJDK 64-Bit Server VM (build 21.0.12.1+1-1-24.04.4-Ubuntu, mixed mode, sharing)


In [11]:
!wget -q https://archive.apache.org/dist/spark/spark-3.5.1/spark-3.5.1-bin-hadoop3.tgz
!tar xf spark-3.5.1-bin-hadoop3.tgz
!ls spark-3.5.1-bin-hadoop3

bin   data	jars	    LICENSE   NOTICE  R		 RELEASE  yarn
conf  examples	kubernetes  licenses  python  README.md  sbin


In [12]:
import os
os.environ["SPARK_HOME"] = "/content/spark-3.5.1-bin-hadoop3"
os.environ["PATH"] += ":/content/spark-3.5.1-bin-hadoop3/bin:/content/spark-3.5.1-bin-hadoop3/sbin"

Kiểm tra

In [5]:
!spark-submit --version

/usr/local/bin/spark-submit: line 27: /content/spark-3.5.1-bin-hadoop3/bin/spark-class: No such file or directory


2. Dựng cluster Master + 2 Worker

In [15]:
import os
os.environ["SPARK_MASTER_HOST"] = "127.0.0.1"
os.environ["SPARK_WORKER_INSTANCES"] = "2"   # 2 worker
os.environ["SPARK_WORKER_CORES"] = "1"
os.environ["SPARK_WORKER_MEMORY"] = "1g"

!start-master.sh
!start-worker.sh spark://127.0.0.1:7077

starting org.apache.spark.deploy.master.Master, logging to /content/spark-3.5.1-bin-hadoop3/logs/spark--org.apache.spark.deploy.master.Master-1-5b5efab95355.out
starting org.apache.spark.deploy.worker.Worker, logging to /content/spark-3.5.1-bin-hadoop3/logs/spark--org.apache.spark.deploy.worker.Worker-1-5b5efab95355.out
starting org.apache.spark.deploy.worker.Worker, logging to /content/spark-3.5.1-bin-hadoop3/logs/spark--org.apache.spark.deploy.worker.Worker-2-5b5efab95355.out


In [17]:
!curl -s http://localhost:8080/json/ | python -c "import sys,json; d=json.load(sys.stdin); print('Trạng thái:', d['status']); print('Số worker:', len(d['workers']))"

Traceback (most recent call last):
  File "<string>", line 1, in <module>
    import sys,json; d=json.load(sys.stdin); print('Trạng thái:', d['status']); print('Số worker:', len(d['workers']))
                       ~~~~~~~~~^^^^^^^^^^^
  File "/usr/lib/python3.13/json/__init__.py", line 298, in load
    return loads(fp.read(),
        cls=cls, object_hook=object_hook,
        parse_float=parse_float, parse_int=parse_int,
        parse_constant=parse_constant, object_pairs_hook=object_pairs_hook, **kw)
  File "/usr/lib/python3.13/json/__init__.py", line 352, in loads
    return _default_decoder.decode(s)
           ~~~~~~~~~~~~~~~~~~~~~~~^^^
  File "/usr/lib/python3.13/json/decoder.py", line 345, in decode
    obj, end = self.raw_decode(s, idx=_w(s, 0).end())
               ~~~~~~~~~~~~~~~^^^^^^^^^^^^^^^^^^^^^^^
  File "/usr/lib/python3.13/json/decoder.py", line 363, in raw_decode
    raise JSONDecodeError("Expecting value", s, err.value) from None
json.decoder.JSONDecodeError: Expecti

In [20]:
# Chạy lệnh này để xem chi tiết phản hồi từ Web UI của Spark Master
!curl http://localhost:8080/json/

In [22]:
# Kiểm tra file log của Spark Master để tìm cổng Web UI chính xác
!cat /content/spark-3.5.1-bin-hadoop3/logs/spark--org.apache.spark.deploy.master.Master-*.out | grep "Started WebUI"

3. RDD trên Scala

Ví dụ 1: Tính toán cơ bản

In [28]:
%%writefile vd1.scala
val rdd = spark.sparkContext.parallelize(List(1,2,3,4,5))
println(s"SUM: ${rdd.sum()}")
println(s"MIN: ${rdd.min()}")
println(s"MAX: ${rdd.max()}")

Overwriting vd1.scala


In [29]:
!spark-shell --master local[*] < vd1.scala 2>/dev/null

[0.004s][warning][os,container] Cgroup memory controller path at '/sys/fs/cgroup' seems to have moved to '/../../jupyter-children', detected limits won't be accurate
[0.004s][warning][os,container] Cgroup cpu controller path at '/sys/fs/cgroup' seems to have moved to '/../../jupyter-children', detected limits won't be accurate
[0.001s][warning][os,container] Cgroup memory controller path at '/sys/fs/cgroup' seems to have moved to '/../../jupyter-children', detected limits won't be accurate
[0.001s][warning][os,container] Cgroup cpu controller path at '/sys/fs/cgroup' seems to have moved to '/../../jupyter-children', detected limits won't be accurate
Spark context Web UI available at http://5b5efab95355:4040
Spark context available as 'sc' (master = local[*], app id = local-1795244963368).
Spark session available as 'spark'.
Welcome to
      ____              __
     / __/__  ___ _____/ /__
    _\ \/ _ \/ _ `/ __/  '_/
   /___/ .__/\_,_/_/ /_/\_\   version 3.5.1
      /_/
         
Usin

Ví dụ 2: Loại bỏ trùng lặp

In [42]:
%%writefile vd2.scala
val rdd = spark.sparkContext.parallelize(List("Germany India USA", "USA India Russia"))
val unique = rdd.flatMap(_.split(" ")).distinct()
unique.collect().foreach(println)

Overwriting vd2.scala


In [43]:
!spark-shell --master local[*] < vd2.scala 2>/dev/null

[0.001s][warning][os,container] Cgroup memory controller path at '/sys/fs/cgroup' seems to have moved to '/../../jupyter-children', detected limits won't be accurate
[0.001s][warning][os,container] Cgroup cpu controller path at '/sys/fs/cgroup' seems to have moved to '/../../jupyter-children', detected limits won't be accurate
[0.001s][warning][os,container] Cgroup memory controller path at '/sys/fs/cgroup' seems to have moved to '/../../jupyter-children', detected limits won't be accurate
[0.001s][warning][os,container] Cgroup cpu controller path at '/sys/fs/cgroup' seems to have moved to '/../../jupyter-children', detected limits won't be accurate
Spark context Web UI available at http://5b5efab95355:4041
Spark context available as 'sc' (master = local[*], app id = local-1795245975952).
Spark session available as 'spark'.
Welcome to
      ____              __
     / __/__  ___ _____/ /__
    _\ \/ _ \/ _ `/ __/  '_/
   /___/ .__/\_,_/_/ /_/\_\   version 3.5.1
      /_/
         
Usin

Ví dụ 3: Đếm số lần xuất hiện của mỗi từ (WordCount RDD)

In [41]:
%%writefile vd3.scala
val data = List("Spark is fast", "Spark is powerful", "Hadoop and Spark")
val rdd = spark.sparkContext.parallelize(data)
val wordCount = rdd.flatMap(_.split(" "))
  .map(word => (word.toLowerCase, 1))
  .reduceByKey(_ + _)
wordCount.collect().foreach(println)

Overwriting vd3.scala


In [44]:
!spark-shell --master local[*] < vd3.scala 2>/dev/null

[0.001s][warning][os,container] Cgroup memory controller path at '/sys/fs/cgroup' seems to have moved to '/../../jupyter-children', detected limits won't be accurate
[0.001s][warning][os,container] Cgroup cpu controller path at '/sys/fs/cgroup' seems to have moved to '/../../jupyter-children', detected limits won't be accurate
[0.001s][warning][os,container] Cgroup memory controller path at '/sys/fs/cgroup' seems to have moved to '/../../jupyter-children', detected limits won't be accurate
[0.001s][warning][os,container] Cgroup cpu controller path at '/sys/fs/cgroup' seems to have moved to '/../../jupyter-children', detected limits won't be accurate
Spark context Web UI available at http://5b5efab95355:4041
Spark context available as 'sc' (master = local[*], app id = local-1795246005609).
Spark session available as 'spark'.
Welcome to
      ____              __
     / __/__  ___ _____/ /__
    _\ \/ _ \/ _ `/ __/  '_/
   /___/ .__/\_,_/_/ /_/\_\   version 3.5.1
      /_/
         
Usin

Ví dụ 4: Đọc file dữ liệu từ HDFS hoặc thư mục cục bộ

In [45]:
%%writefile students.csv
name,age,major
An,21,IT
Binh,22,Data
Chi,21,AI

Writing students.csv


In [46]:
%%writefile vd4.scala
val df = spark.read.option("header", "true").csv("/content/students.csv")
println("================ ORIGINAL DF ================")
df.show()
val dfFiltered = df.filter($"major" === "Data")
println("================ FILTERED DF ================")
dfFiltered.show()
println("=========================================")

Writing vd4.scala


In [47]:
!spark-shell --master local[*] < vd4.scala 2>/dev/null

[0.001s][warning][os,container] Cgroup memory controller path at '/sys/fs/cgroup' seems to have moved to '/../../jupyter-children', detected limits won't be accurate
[0.001s][warning][os,container] Cgroup cpu controller path at '/sys/fs/cgroup' seems to have moved to '/../../jupyter-children', detected limits won't be accurate
[0.001s][warning][os,container] Cgroup memory controller path at '/sys/fs/cgroup' seems to have moved to '/../../jupyter-children', detected limits won't be accurate
[0.001s][warning][os,container] Cgroup cpu controller path at '/sys/fs/cgroup' seems to have moved to '/../../jupyter-children', detected limits won't be accurate
Spark context Web UI available at http://5b5efab95355:4041
Spark context available as 'sc' (master = local[*], app id = local-1795246091758).
Spark session available as 'spark'.
Welcome to
      ____              __
     / __/__  ___ _____/ /__
    _\ \/ _ \/ _ `/ __/  '_/
   /___/ .__/\_,_/_/ /_/\_\   version 3.5.1
      /_/
         
Usin

Ví dụ 5: Tính thống kê cơ bản (mean, max, count)

In [48]:
%%writefile vd5.scala
val scores = spark.sparkContext.parallelize(List(9,7,6,10,8,5,9))
println("================ RESULT ================")
println("Tổng điểm: " + scores.sum())
println("Trung bình: " + scores.mean())
println("Số sinh viên >= 8: " + scores.filter(_ >= 8).count())
println("========================================")

Writing vd5.scala


In [49]:
!spark-shell --master local[*] < vd5.scala 2>/dev/null

[0.001s][warning][os,container] Cgroup memory controller path at '/sys/fs/cgroup' seems to have moved to '/../../jupyter-children', detected limits won't be accurate
[0.001s][warning][os,container] Cgroup cpu controller path at '/sys/fs/cgroup' seems to have moved to '/../../jupyter-children', detected limits won't be accurate
[0.001s][warning][os,container] Cgroup memory controller path at '/sys/fs/cgroup' seems to have moved to '/../../jupyter-children', detected limits won't be accurate
[0.001s][warning][os,container] Cgroup cpu controller path at '/sys/fs/cgroup' seems to have moved to '/../../jupyter-children', detected limits won't be accurate
Spark context Web UI available at http://5b5efab95355:4041
Spark context available as 'sc' (master = local[*], app id = local-1795246139012).
Spark session available as 'spark'.
Welcome to
      ____              __
     / __/__  ___ _____/ /__
    _\ \/ _ \/ _ `/ __/  '_/
   /___/ .__/\_,_/_/ /_/\_\   version 3.5.1
      /_/
         
Usin

4. PySpark

In [32]:
from pyspark.sql import SparkSession

spark = SparkSession.builder \
    .appName("Spark Practice") \
    .master("local[*]") \
    .getOrCreate()
spark

Ví dụ 6: Đếm số lần xuất hiện của mỗi từ

In [33]:
from pyspark.sql import SparkSession
spark = SparkSession.builder.master("local[*]").appName("WordCount").getOrCreate()
rdd = spark.sparkContext.parallelize(["Big data with Spark", "Spark and Python"])
result = (rdd.flatMap(lambda x: x.split(" "))
             .map(lambda w: (w.lower(), 1))
             .reduceByKey(lambda a, b: a + b))
for x in result.collect():
    print(x)

('big', 1)
('with', 1)
('and', 1)
('python', 1)
('data', 1)
('spark', 2)


Ví dụ 7: Bài toán cộng 10 triệu số, một lần bằng Python thường, một lần bằng Spark,
so sánh thời gian chạy.

In [35]:
import time
from pyspark.sql import SparkSession

# Python thường
t1 = time.time()
sum([i for i in range(10_000_000)])
t2 = time.time()
print("Python:", t2 - t1, "giây")

# Spark
spark = SparkSession.builder.master("local[*]").appName("SpeedTest").getOrCreate()
t3 = time.time()
spark.sparkContext.parallelize(range(10_000_000)).sum()
t4 = time.time()
print("Spark:", t4 - t3, "giây")

Python: 0.7878541946411133 giây
Spark: 0.45496702194213867 giây


Ví dụ 8: WordCount (RDD + DataFrame API) chạy trực tiếp trên cụm Spark dựng
bằng Docker Compose

In [50]:
!mkdir -p /content/data/wordcount /content/apps

In [51]:
%%writefile /content/data/wordcount/doc1.txt
Spark is fast and Spark is fun

Writing /content/data/wordcount/doc1.txt


In [52]:
%%writefile /content/data/wordcount/doc2.txt
Hadoop and Spark are popular for big data

Writing /content/data/wordcount/doc2.txt


In [53]:
%%writefile /content/apps/wordcount.py
from pyspark.sql import SparkSession
from pyspark.sql.functions import explode, split, lower, col, count as spark_count

spark = SparkSession.builder.appName('WordCount').getOrCreate()

# RDD API
rdd = spark.sparkContext.textFile('/content/data/wordcount')
counts_rdd = (rdd.flatMap(lambda line: line.split())
                 .map(lambda w: (w.lower(), 1))
                 .reduceByKey(lambda a, b: a + b))
print('=== Top via RDD ===')
for w, c in counts_rdd.takeOrdered(10, key=lambda x: -x[1]):
    print(w, c)

# DataFrame API
df = spark.read.text('/content/data/wordcount')
words = explode(split(col('value'), r'\s+')).alias('word')
wc_df = (df.select(lower(words).alias('word'))
           .groupBy('word').agg(spark_count('*').alias('cnt'))
           .orderBy(col('cnt').desc()))
wc_df.show(10, truncate=False)

# Lưu kết quả ra CSV
wc_df.coalesce(1).write.mode('overwrite').option('header', True).csv('/content/data/wordcount_out')
print('Saved to /content/data/wordcount_out')
spark.stop()

Writing /content/apps/wordcount.py


In [56]:
%%writefile /content/apps/wordcount.py
from pyspark.sql import SparkSession
from pyspark.sql.functions import explode, split, lower, col, count as spark_count

spark = SparkSession.builder.appName('WordCount').getOrCreate()

# RDD API
rdd = spark.sparkContext.textFile('/content/data/wordcount')
counts_rdd = (rdd.flatMap(lambda line: line.split())
                 .map(lambda w: (w.lower(), 1))
                 .reduceByKey(lambda a, b: a + b))
print('=== Top via RDD ===')
for w, c in counts_rdd.takeOrdered(10, key=lambda x: -x[1]):
    print(w, c)

# DataFrame API (Đã sửa lỗi lồng generator)
df = spark.read.text('/content/data/wordcount')

# Bước 1: Explode trước để tách dòng thành các từ
exploded_df = df.select(explode(split(col('value'), r'\s+')).alias('raw_word'))

# Bước 2: Apply hàm lower() trên cột từ đã tách
wc_df = (exploded_df.select(lower(col('raw_word')).alias('word'))
           .groupBy('word').agg(spark_count('*').alias('cnt'))
           .orderBy(col('cnt').desc()))

wc_df.show(10, truncate=False)

# Lưu kết quả ra CSV
wc_df.coalesce(1).write.mode('overwrite').option('header', True).csv('/content/data/wordcount_out')
print('Saved to /content/data/wordcount_out')
spark.stop()

Overwriting /content/apps/wordcount.py


In [57]:
!spark-submit --master spark://127.0.0.1:7077 /content/apps/wordcount.py 2>/dev/null

[0.001s][warning][os,container] Cgroup memory controller path at '/sys/fs/cgroup' seems to have moved to '/../../jupyter-children', detected limits won't be accurate
[0.001s][warning][os,container] Cgroup cpu controller path at '/sys/fs/cgroup' seems to have moved to '/../../jupyter-children', detected limits won't be accurate
[0.001s][warning][os,container] Cgroup memory controller path at '/sys/fs/cgroup' seems to have moved to '/../../jupyter-children', detected limits won't be accurate
[0.001s][warning][os,container] Cgroup cpu controller path at '/sys/fs/cgroup' seems to have moved to '/../../jupyter-children', detected limits won't be accurate
=== Top via RDD ===
spark 3
and 2
is 2
hadoop 1
for 1
fast 1
fun 1
are 1
popular 1
big 1
+-------+---+
|word   |cnt|
+-------+---+
|spark  |3  |
|is     |2  |
|and    |2  |
|fast   |1  |
|fun    |1  |
|popular|1  |
|for    |1  |
|data   |1  |
|are    |1  |
|hadoop |1  |
+-------+---+
only showing top 10 rows

Saved to /content/data/wordcoun

In [58]:
!ls /content/data/wordcount_out
!cat /content/data/wordcount_out/part-*.csv

part-00000-9c17aedc-6da0-4e0c-90eb-c946cde2536d-c000.csv  _SUCCESS
word,cnt
spark,3
and,2
is,2
popular,1
for,1
data,1
are,1
hadoop,1
big,1
fast,1
fun,1
